# Taro carotenoid pathway — Part 1 exploratory figures

Exploratory plots for the Part 1 results. These are **not** manuscript
figures. Each cell produces one plot, so you can look at it, decide whether
it says anything, and change it. Composite panels and final styling come
later, once we know which of these are worth keeping.

Run in the `ibnu` environment. Everything here reads text files
(TSV, newick) produced by `scripts/01_` through `scripts/16_`, so no
bioinformatics tooling is needed — pandas, matplotlib, seaborn, scipy
and Biopython are enough.

## A note on statistics

There is a temptation to put a p-value on "taro has fewer PSY genes than
other species". Resist it. The ten species are tips on a phylogeny, not
independent samples: rice and maize share a count because they share an
ancestor, not because they are two independent draws. Applying a t-test or
chi-square here is phylogenetic pseudoreplication, a well-documented error
in comparative genomics.

With n = 10 and strong phylogenetic structure, the defensible treatment is
descriptive — show the counts, show the tree, and let the reader see the
non-independence. Phylogenetically independent contrasts exist for
continuous traits, but with ten tips and counts of 1 to 3 they would be
overkill and still underpowered.

Tests are used below only where observations genuinely are independent:
pairwise distances within a gene family, and protein length distributions.

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from Bio import Phylo
from scipy import stats

# Paths. Adjust STORE if the drive letter differs.
CODE = Path.home() / "Code" / "taro"
STORE = Path("/mnt/f/RA/Downstream/Project5_TARO")
TREES = STORE / "work" / "05_orthology" / "trees"
TABLES = CODE / "results" / "tables"
FIGS = CODE / "results" / "figures"
FIGS.mkdir(parents=True, exist_ok=True)

# Plotting defaults. Genomics figures favour clarity over decoration:
# thin lines, no chartjunk, colour used only where it carries information.
plt.rcParams.update({
    "figure.dpi": 110,
    "savefig.dpi": 300,
    "savefig.bbox": "tight",
    "font.size": 10,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "grid.linewidth": 0.5,
})

SPECIES = {
    "colesc": ("Colocasia esculenta", "taro", "Alismatales"),
    "zosmar": ("Zostera marina", "eelgrass", "Alismatales"),
    "musacu": ("Musa acuminata", "banana", "Zingiberales"),
    "orysat": ("Oryza sativa", "rice", "Poales"),
    "zeamay": ("Zea mays", "maize", "Poales"),
    "manesc": ("Manihot esculenta", "cassava", "Malpighiales"),
    "aratha": ("Arabidopsis thaliana", "thale cress", "Brassicales"),
    "soltub": ("Solanum tuberosum", "potato", "Solanales"),
    "nelnuc": ("Nelumbo nucifera", "lotus", "Proteales"),
    "ambtri": ("Amborella trichopoda", "amborella", "Amborellales"),
}

# Highlight scheme used throughout: taro is the subject, cassava is the
# comparison that matters to the collaboration, everything else is context.
C_TARO, C_CASSAVA, C_OTHER = "#e29c94", "#78a6c5", "#95a5a6"

print(f"code   {CODE}")
print(f"trees  {TREES}")
print(f"figs   {FIGS}")

## 1. Verified PSY copy number

From `16_verify_counts.sh`, counting only sequences inside the clade that
contains exactly one Arabidopsis sequence. Arabidopsis has a single PSY, so
once a clade picks up a second Arabidopsis gene it has expanded into the
wider squalene and phytoene synthase superfamily.

Species are ordered by phylogeny rather than by count, because the ordering
is the argument: the two single-copy species sit together.

In [ ]:
psy = pd.DataFrame([
    # tag,      verified PSY,  naive hit count
    ("ambtri", 2, 3), ("nelnuc", 2, 2), ("soltub", 3, 4),
    ("aratha", 1, 1), ("manesc", 2, 3), ("zosmar", 1, 1),
    ("colesc", 1, 1), ("musacu", 3, 3), ("orysat", 2, 3),
    ("zeamay", 2, 3),
], columns=["tag", "psy", "hits"])

psy["species"] = psy.tag.map(lambda t: SPECIES[t][0])
psy["common"] = psy.tag.map(lambda t: SPECIES[t][1])
psy["order"] = psy.tag.map(lambda t: SPECIES[t][2])

# rough phylogenetic ordering, basal to derived, monocots grouped
phylo_order = ["ambtri", "nelnuc", "soltub", "aratha", "manesc",
               "zosmar", "colesc", "musacu", "orysat", "zeamay"]
psy = psy.set_index("tag").loc[phylo_order].reset_index()

fig, ax = plt.subplots(figsize=(7, 4.2))
colors = [C_TARO if t == "colesc" else C_CASSAVA if t == "manesc" else C_OTHER
          for t in psy.tag]
bars = ax.barh(range(len(psy)), psy.psy, color=colors, height=0.68)

ax.set_yticks(range(len(psy)))
ax.set_yticklabels([f"{r.species}\n({r.common})" for r in psy.itertuples()],
                   fontsize=8.5)
ax.invert_yaxis()
ax.set_xlabel("phytoene synthase genes (verified by tree topology)")
ax.set_xticks(range(0, 5))
ax.grid(axis="y", visible=False)

for i, v in enumerate(psy.psy):
    ax.text(v + 0.06, i, str(v), va="center", fontsize=9)

# mark the Alismatales, the clade containing both single-copy species
alis = [i for i, t in enumerate(psy.tag) if SPECIES[t][2] == "Alismatales"]
ax.axhspan(min(alis) - 0.45, max(alis) + 0.45,
           color="#f39c12", alpha=0.12, zorder=0)
ax.text(3.6, np.mean(alis), "Alismatales", rotation=90,
        va="center", ha="center", fontsize=8, color="#b9770e")

ax.set_title("Taro carries one PSY; cassava, the transgene source, carries two",
             fontsize=10.5, loc="left")
plt.tight_layout()
plt.savefig(FIGS / "01_psy_copy_number.png")
plt.show()

Both single-copy species are Alismatales. Two independent lineages at one
copy, against two or three everywhere else, suggests a lineage-level trait
rather than something specific to taro.

This cannot be confirmed with two tips. A third Alismatid would settle it,
and *Spirodela polyrhiza* is the obvious candidate — which is why its absence
from the species set is the project's main evolutionary limitation rather
than a generic caveat.

## 2. What the naive count would have said

Worth plotting once, because it shows how much the verification step
changed. Counting homology search hits rather than clade membership
inflates five of the ten species, including cassava, where it would have
produced three PSY genes against the two the literature describes.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
x = np.arange(len(psy))
w = 0.38
ax.bar(x - w/2, psy.hits, w, label="homology hits (naive)",
       color="#bdc3c7", edgecolor="none")
ax.bar(x + w/2, psy.psy, w, label="inside PSY clade (verified)",
       color="#34495e", edgecolor="none")

ax.set_xticks(x)
ax.set_xticklabels(psy.common, rotation=40, ha="right", fontsize=8.5)
ax.set_ylabel("gene count")
ax.legend(frameon=False, fontsize=8.5)
ax.grid(axis="x", visible=False)

changed = psy.hits != psy.psy
for i in np.where(changed)[0]:
    ax.text(i, max(psy.hits[i], psy.psy[i]) + 0.12, "corrected",
            ha="center", fontsize=7, color="#c0392b")

ax.set_title("Hit counting overestimates copy number in 5 of 10 species",
             fontsize=10.5, loc="left")
plt.tight_layout()
plt.savefig(FIGS / "02_naive_vs_verified.png")
plt.show()

## 3. PSY gene tree

Nineteen sequences, the clade containing exactly one Arabidopsis gene.
Taro and cassava are highlighted.

Rerun FastTree without `-nosupport` first so the nodes carry SH-like local
support. Without it, a claim about clade membership has no stated
confidence, which is the first thing anyone will ask about.

In [ ]:
tree_file = TREES / "psy_support.tree"
if not tree_file.exists():
    tree_file = TREES / "psy.tree"
    print("WARNING: using the tree without support values.")
    print("Run:  fasttree -quiet < psy.aln > psy_support.tree")

t = Phylo.read(tree_file, "newick")
t.root_at_midpoint()
t.ladderize()


def pretty(clade):
    """Readable tip labels; blank for internal nodes."""
    if not clade.name:
        return ""
    tag, acc = clade.name.split("_", 1)
    sp, common, _ = SPECIES.get(tag, (tag, "", ""))
    return f"{sp}  {acc}"


fig, ax = plt.subplots(figsize=(8, 7))
Phylo.draw(t, axes=ax, do_show=False, label_func=pretty,
           show_confidence=False,
           label_colors=lambda lbl: (C_TARO if "Colocasia" in lbl
                                     else C_CASSAVA if "Manihot" in lbl
                                     else "#2c3e50"))
ax.set_xlabel("substitutions per site")
ax.set_ylabel("")
ax.set_yticks([])
ax.grid(False)
ax.set_title("Phytoene synthase gene tree", fontsize=10.5, loc="left")
plt.tight_layout()
plt.savefig(FIGS / "03_psy_tree.png")
plt.show()

## 4. Support values, if present

A tree figure without support is a drawing, not evidence. This plots the
distribution of node support so you can say how well resolved the topology
is rather than implying it is certain.

In [ ]:
conf = [c.confidence for c in t.get_nonterminals()
        if c.confidence is not None]

if conf:
    fig, ax = plt.subplots(figsize=(5.5, 3.2))
    ax.hist(conf, bins=np.linspace(0, 1, 21),
            color="#34495e", edgecolor="white", linewidth=0.6)
    ax.axvline(0.7, color=C_TARO, ls="--", lw=1.2)
    ax.text(0.705, ax.get_ylim()[1] * 0.9, " 0.7 conventional threshold",
            fontsize=8, color=C_TARO)
    ax.set_xlabel("SH-like local support")
    ax.set_ylabel("nodes")
    ax.set_title(f"{sum(c >= 0.7 for c in conf)}/{len(conf)} nodes at or above 0.7",
                 fontsize=10.5, loc="left")
    plt.tight_layout()
    plt.savefig(FIGS / "04_psy_support.png")
    plt.show()
else:
    print("No support values on this tree. Rerun FastTree without -nosupport:")
    print(f"  cd {TREES}")
    print("  fasttree -quiet < psy.aln > psy_support.tree")

## 5. CCD4 and NCED separate cleanly

Three taro genes were called CCD4 orthologs, but two of them also matched
NCED3. CCD and NCED belong to the same superfamily, so the assignment had to
be checked against topology.

Here the two anchors are Arabidopsis AT4G19170 (CCD4) and AT3G14440 (NCED3),
and each taro gene is placed by its patristic distance to both.

A statistical test is defensible in this panel, unlike the copy-number
comparison. The observations are pairwise distances within one tree, and the
question is whether the two groups differ — a Mann-Whitney U is appropriate
because the distances are not normally distributed. Note that the separation
is large enough to be obvious by eye, so the test confirms rather than
discovers.

In [ ]:
ccd = pd.DataFrame([
    ("Ces13251", 0.966, 2.133), ("Ces13250", 1.194, 2.362),
    ("Ces03723", 1.309, 2.477), ("Ces11733", 2.056, 0.808),
    ("Ces27429", 2.059, 0.810), ("Ces09230", 2.071, 0.823),
    ("Ces28200", 2.141, 0.893), ("Ces16312", 2.142, 0.894),
    ("Ces10063", 2.107, 1.196), ("Ces00245", 2.479, 2.324),
], columns=["gene", "d_ccd4", "d_nced3"])
ccd["call"] = np.where(ccd.d_ccd4 < ccd.d_nced3, "CCD4", "NCED")
ccd["margin"] = (ccd.d_nced3 - ccd.d_ccd4).abs()

fig, ax = plt.subplots(figsize=(6, 5.2))
for call, c, m in [("CCD4", "#c0392b", "o"), ("NCED", "#2874a6", "s")]:
    s = ccd[ccd.call == call]
    ax.scatter(s.d_ccd4, s.d_nced3, c=c, marker=m, s=70,
               edgecolor="white", linewidth=1, label=call, zorder=3)
    for r in s.itertuples():
        ax.annotate(r.gene, (r.d_ccd4, r.d_nced3), fontsize=7.5,
                    xytext=(6, -3), textcoords="offset points")

lim = [0.6, 2.7]
ax.plot(lim, lim, ls="--", c="#7f8c8d", lw=1, zorder=1)
ax.text(2.5, 2.56, "equidistant", fontsize=7.5, rotation=45,
        color="#7f8c8d", ha="right")
ax.set_xlim(lim); ax.set_ylim(lim)
ax.set_xlabel("patristic distance to AtCCD4 (AT4G19170)")
ax.set_ylabel("patristic distance to AtNCED3 (AT3G14440)")
ax.legend(frameon=False, title="assignment", fontsize=9)
ax.set_title("Each taro gene is ~2x closer to one anchor than the other",
             fontsize=10.5, loc="left")
plt.tight_layout()
plt.savefig(FIGS / "05_ccd_nced_assignment.png")
plt.show()

u, p = stats.mannwhitneyu(ccd[ccd.call == "CCD4"].margin,
                          ccd[ccd.call == "NCED"].margin)
print(f"CCD4 group n={sum(ccd.call=='CCD4')}, NCED group n={sum(ccd.call=='NCED')}")
print(f"Mann-Whitney U on assignment margin: U={u:.1f}, p={p:.3f}")
print("\nNote Ces00245 — nearly equidistant (margin "
      f"{ccd[ccd.gene=='Ces00245'].margin.iloc[0]:.3f}). It is CCD1, not CCD4,")
print("and sits outside both anchors. Worth excluding or labelling separately.")

## 6. Pathway completeness

The full inventory from `11_pathway_homology.sh`. A heatmap makes the shape
of the result visible at once: most steps single copy, a few expanded.

Read it as presence and copy number, not activity. Nothing here measures
expression, and a gene present in the genome may be silent in corm tissue.

In [ ]:
f = TABLES / "pathway_homology.tsv"
ph = pd.read_csv(f, sep="\t")

conf = ph[ph.call == "ortholog (RBH)"]
counts = (conf.groupby(["step", "gene"]).size()
          .reset_index(name="n").sort_values(["step", "gene"]))

step_order = ["MEP", "backbone", "core", "cyclase", "xanth", "degrade", "sink"]
counts["step"] = pd.Categorical(counts.step, step_order, ordered=True)
counts = counts.sort_values(["step", "gene"])

fig, ax = plt.subplots(figsize=(9, 2.6))
colors = []
for r in counts.itertuples():
    if r.gene == "PSY":
        colors.append(C_TARO)
    elif r.gene == "CCD4":
        colors.append("#e67e22")
    elif r.n > 1:
        colors.append("#34495e")
    else:
        colors.append("#bdc3c7")

ax.bar(range(len(counts)), counts.n, color=colors, width=0.72)
ax.set_xticks(range(len(counts)))
ax.set_xticklabels(counts.gene, rotation=70, ha="right", fontsize=8)
ax.set_ylabel("taro genes")
ax.set_yticks(range(0, counts.n.max() + 2))
ax.grid(axis="x", visible=False)

# step boundaries
pos = 0
for s in step_order:
    n = (counts.step == s).sum()
    if n:
        ax.axvline(pos - 0.5, color="#ecf0f1", lw=1, zorder=0)
        ax.text(pos + n/2 - 0.5, counts.n.max() + 1.2, s,
                ha="center", fontsize=8, color="#7f8c8d")
        pos += n

ax.set_title("Carotenoid and MEP pathway in taro, one-to-one orthologs",
             fontsize=10.5, loc="left")
plt.tight_layout()
plt.savefig(FIGS / "06_pathway_inventory.png")
plt.show()

print("\nExpanded steps:")
for r in counts[counts.n > 1].itertuples():
    print(f"  {r.gene:<10} {r.n}")

## 7. The split gene model

Ces12496 and Ces12497 matched PSY but covered only 26% and 34% of the query.
They lie 259 bp apart on the same strand with nothing annotated between
them, and their alignments cover non-overlapping regions.

Compare with Ces13250 and Ces13251, the two tandem CCD4 genes: 75 kb apart,
both encoding substantial proteins. Same adjacency signal, different answer.
Worth plotting side by side, since it is the distinction that separates an
annotation artifact from a real duplication.

In [ ]:
loci = pd.DataFrame([
    ("Ces24605", "Superscaffold13", 96_419_008, 96_426_240, "-", 428, "PSY, intact"),
    ("Ces12496", "Superscaffold7",  79_028_474, 79_030_838, "+", 112, "PSY-like fragment"),
    ("Ces12497", "Superscaffold7",  79_031_097, 79_032_356, "+", 155, "PSY-like fragment"),
    ("Ces13250", "Superscaffold7",  99_656_322, 99_660_230, "+", 456, "CCD4"),
    ("Ces13251", "Superscaffold7",  99_735_303, 99_737_395, "+", 630, "CCD4"),
    ("Ces03723", "Superscaffold2", 192_445_091, 192_451_381, "-", 892, "CCD4"),
], columns=["gene", "scaffold", "start", "end", "strand", "aa", "note"])

fig, axes = plt.subplots(1, 2, figsize=(10, 3))

# left: the PSY fragments
a = axes[0]
sub = loci[loci.gene.isin(["Ces12496", "Ces12497"])]
origin = sub.start.min()
for i, r in enumerate(sub.itertuples()):
    a.barh(0, r.end - r.start, left=r.start - origin, height=0.4,
           color=C_TARO, alpha=0.85)
    a.text((r.start + r.end)/2 - origin, 0.32, f"{r.gene}\n{r.aa} aa",
           ha="center", fontsize=7.5)
gap = sub.start.max() - sub.end.min()
a.annotate("", xy=(sub.end.min() - origin, -0.3),
           xytext=(sub.start.max() - origin, -0.3),
           arrowprops=dict(arrowstyle="<->", color="#7f8c8d", lw=1))
a.text((sub.end.min() + sub.start.max())/2 - origin, -0.47,
       f"{gap} bp", ha="center", fontsize=8, color="#7f8c8d")
a.set_ylim(-0.8, 0.8); a.set_yticks([])
a.set_xlabel("position on Superscaffold7 (bp from first gene)")
a.set_title("PSY-like locus: one gene, two models\n112 + 155 = 267 aa of 437",
            fontsize=9.5, loc="left")
a.grid(axis="y", visible=False)

# right: the tandem CCD4 pair
b = axes[1]
sub = loci[loci.gene.isin(["Ces13250", "Ces13251"])]
origin = sub.start.min()
for r in sub.itertuples():
    b.barh(0, r.end - r.start, left=r.start - origin, height=0.4,
           color="#e67e22", alpha=0.85)
    b.text((r.start + r.end)/2 - origin, 0.32, f"{r.gene}\n{r.aa} aa",
           ha="center", fontsize=7.5)
gap = sub.start.max() - sub.end.min()
b.annotate("", xy=(sub.end.min() - origin, -0.3),
           xytext=(sub.start.max() - origin, -0.3),
           arrowprops=dict(arrowstyle="<->", color="#7f8c8d", lw=1))
b.text((sub.end.min() + sub.start.max())/2 - origin, -0.47,
       f"{gap:,} bp", ha="center", fontsize=8, color="#7f8c8d")
b.set_ylim(-0.8, 0.8); b.set_yticks([])
b.set_xlabel("position on Superscaffold7 (bp from first gene)")
b.set_title("CCD4 tandem pair: two genes\nboth near full length",
            fontsize=9.5, loc="left")
b.grid(axis="y", visible=False)

plt.tight_layout()
plt.savefig(FIGS / "07_locus_structure.png")
plt.show()

## 8. Protein length as a fragmentation signal

If the annotation systematically truncates gene models, pathway proteins
should be short relative to their Arabidopsis counterparts. Plotting the
ratio shows whether the two PSY fragments are unusual or whether truncation
is widespread — which bears directly on whether Part 2 is worth doing.

A one-sample Wilcoxon test against a ratio of 1 is appropriate here: each
gene is an independent observation within one genome, and the question is
whether the median ratio departs from parity.

In [ ]:
ph2 = ph[ph.call.isin(["ortholog (RBH)", "partial (RBH)"])].copy()
ph2 = ph2.dropna(subset=["coverage_pct"])

fig, ax = plt.subplots(figsize=(7, 3.6))
sns.stripplot(data=ph2, x="coverage_pct", y="step", hue="call",
              palette={"ortholog (RBH)": "#34495e", "partial (RBH)": C_TARO},
              size=6, alpha=0.85, ax=ax, jitter=0.18)
ax.axvline(70, ls="--", c="#7f8c8d", lw=1)
ax.text(71, -0.45, "70% threshold", fontsize=7.5, color="#7f8c8d")
ax.set_xlabel("alignment coverage of the Arabidopsis query (%)")
ax.set_ylabel("")
ax.legend(frameon=False, fontsize=8, loc="lower left")
ax.set_title("Most taro pathway genes align over most of the query",
             fontsize=10.5, loc="left")
plt.tight_layout()
plt.savefig(FIGS / "08_coverage_distribution.png")
plt.show()

cov = ph2[ph2.call == "ortholog (RBH)"].coverage_pct / 100
w, p = stats.wilcoxon(cov - 1)
print(f"n = {len(cov)} one-to-one orthologs")
print(f"median coverage {cov.median()*100:.1f}%")
print(f"Wilcoxon signed-rank against parity: W={w:.1f}, p={p:.4f}")
print("\nInterpretation: a median near or above 100% means coverage is not")
print("systematically truncated, so the two PSY fragments are the exception")
print("rather than the rule. Coverage above 100% occurs where the taro protein")
print("is longer than the Arabidopsis query, which is common and not an error.")

## What to keep

Candidates for the meeting, in order of how quickly they make the point:

1. **Panel 1** — PSY copy number. One glance gives taro versus cassava.
2. **Panel 5** — CCD4 and NCED separation. Shows the assignment is clean.
3. **Panel 7** — split model versus tandem pair. Shows the annotation problem
   concretely, which is the bridge to Parts 2 and 3.

Panels 2, 4, 6 and 8 are method and quality evidence. Useful in the
repository and in a methods section; not needed on a slide.

## What is still missing

- Tree support values, if FastTree has not been rerun. Do this before any
  figure asserting clade membership leaves the repository.
- Expression. Every statement here is about gene presence and copy number.
  Whether the three CCD4 genes are active in corm tissue is unknown, and it
  is the difference between a hypothesis and a finding.
- A third Alismatid. With two tips, the single-copy PSY pattern is a
  suggestion, not an inference.